# dbt (data build tool)

> **适用场景**: 数仓转换层建模、ELT 流程管理
> **面试频率**: ⭐⭐⭐⭐⭐ 极高频（现代数据栈核心工具）

## 目录
1. model 依赖 & ref() 函数
2. Incremental 模型策略
3. Source freshness & Tests
4. Macro & Jinja 模板
5. Snapshots (SCD Type 2)
6. 练习题

---
## 1. model 依赖 & ref() 函数

### dbt 项目结构
```
my_project/
├── models/
│   ├── staging/          # 原始数据清洗层
│   │   └── stg_orders.sql
│   ├── intermediate/     # 中间计算层
│   │   └── int_order_items.sql
│   └── marts/            # 最终业务模型
│       └── fct_orders.sql
├── macros/
├── tests/
└── dbt_project.yml
```

### ref() 函数
`ref()` 是 dbt 模型间依赖的核心，它：
1. 自动解析正确的数据库 schema（dev/prod 环境隔离）
2. 建立 DAG 依赖关系，保证执行顺序
3. 支持跨项目引用（dbt mesh）

```sql
-- models/marts/fct_orders.sql
SELECT
    o.order_id,
    o.customer_id,
    c.customer_name,
    o.total_amount
FROM {{ ref('stg_orders') }} o          -- 引用 staging 层模型
LEFT JOIN {{ ref('stg_customers') }} c   -- 自动建立 DAG 依赖
    ON o.customer_id = c.customer_id
```

### source() 函数
引用原始数据源（非 dbt 管理的表）：
```sql
-- models/staging/stg_orders.sql
SELECT * FROM {{ source('raw', 'orders') }}
-- 映射到 raw.orders 表，并支持 freshness 检查
```

```yaml
# models/staging/sources.yml
sources:
  - name: raw
    schema: raw
    tables:
      - name: orders
        loaded_at_field: _loaded_at
        freshness:
          warn_after: {count: 12, period: hour}
          error_after: {count: 24, period: hour}
```

---
## 2. Incremental 模型策略

### 为什么用 Incremental？
全量刷新大表（TB 级）代价极高，增量模型只处理新增/变更数据。

### 基本语法
```sql
-- models/marts/fct_events.sql
{{ config(
    materialized='incremental',
    unique_key='event_id',
    incremental_strategy='merge'   -- 见下方策略说明
) }}

SELECT
    event_id,
    user_id,
    event_type,
    event_time
FROM {{ ref('stg_events') }}

{% if is_incremental() %}
-- 增量运行时只处理新数据（第一次全量运行）
WHERE event_time > (SELECT MAX(event_time) FROM {{ this }})
{% endif %}
```

### 四种增量策略对比

| 策略 | 原理 | 适用场景 | 支持数仓 |
|------|------|----------|----------|
| `append` | 只追加新行 | 不可变事件流 | 所有 |
| `merge` | MERGE INTO，upsert | 有更新的数据 | BigQuery, Snowflake, Databricks |
| `delete+insert` | 先删后插（按 unique_key）| 无 MERGE 的数仓 | Redshift, Spark |
| `insert_overwrite` | 覆盖特定分区 | 按日期分区的大表 | BigQuery, Spark |

```sql
-- insert_overwrite 示例（按日期分区）
{{ config(
    materialized='incremental',
    incremental_strategy='insert_overwrite',
    partition_by={'field': 'date', 'data_type': 'date'}
) }}
```

### 强制全量刷新
```bash
dbt run --full-refresh --select fct_events
```

---
## 3. Source freshness & Tests

### dbt 内置 Tests（4 种）
```yaml
# models/marts/schema.yml
models:
  - name: fct_orders
    columns:
      - name: order_id
        tests:
          - unique                    # 唯一性
          - not_null                  # 非空
      - name: status
        tests:
          - accepted_values:          # 值域
              values: ['pending', 'shipped', 'delivered', 'cancelled']
      - name: customer_id
        tests:
          - relationships:            # 外键引用完整性
              to: ref('stg_customers')
              field: customer_id
```

### 运行测试
```bash
dbt test                          # 所有测试
dbt test --select fct_orders      # 指定模型
dbt test --select source:raw      # 测试 source
dbt source freshness              # 检查数据源新鲜度
```

### 自定义 Generic Test
```sql
-- tests/generic/positive_value.sql
{% test positive_value(model, column_name) %}
SELECT *
FROM {{ model }}
WHERE {{ column_name }} <= 0
{% endtest %}
```

```yaml
# 使用自定义 test
- name: total_amount
  tests:
    - positive_value
```

---
## 4. Macro & Jinja 模板

### Jinja 基础
```sql
-- 变量
{{ variable_name }}

-- 条件
{% if condition %} ... {% endif %}

-- 循环
{% for item in list %} ... {% endfor %}

-- 注释
{# 这是注释 #}
```

### Macro 定义与使用
```sql
-- macros/cents_to_dollars.sql
{% macro cents_to_dollars(column_name, decimal_places=2) %}
    round({{ column_name }} / 100, {{ decimal_places }})
{% endmacro %}

-- 在模型中使用
SELECT
    order_id,
    {{ cents_to_dollars('amount_cents') }} AS amount_dollars,
    {{ cents_to_dollars('tax_cents', 4) }} AS tax_dollars
FROM {{ ref('stg_orders') }}
```

### 常用内置 Macro
```sql
-- 当前数据库/schema
{{ target.database }}
{{ target.schema }}

-- 运行信息
{{ run_started_at }}
{{ invocation_id }}

-- 动态生成 SQL
{% set payment_methods = ['credit_card', 'paypal', 'wire'] %}
SELECT
    order_id,
    {% for method in payment_methods %}
    SUM(CASE WHEN payment_method = '{{ method }}' THEN amount END) AS {{ method }}_amount
    {% if not loop.last %},{% endif %}
    {% endfor %}
FROM {{ ref('stg_payments') }}
GROUP BY order_id
```

---
## 5. Snapshots (SCD Type 2)

### 什么是 SCD Type 2？
缓慢变化维度 Type 2：当维度数据变化时，**保留历史记录**（新增一行，旧行标记失效）。

### dbt Snapshot 实现
```sql
-- snapshots/customers_snapshot.sql
{% snapshot customers_snapshot %}

{{  config(
    target_schema='snapshots',
    unique_key='customer_id',
    strategy='timestamp',          -- 或 'check'
    updated_at='updated_at'        -- timestamp 策略使用的时间列
) }}

SELECT * FROM {{ source('raw', 'customers') }}

{% endsnapshot %}
```

dbt 自动添加的列：
| 列名 | 含义 |
|------|------|
| `dbt_scd_id` | 快照行唯一 ID |
| `dbt_updated_at` | 该快照记录的更新时间 |
| `dbt_valid_from` | 该版本生效时间 |
| `dbt_valid_to` | 该版本失效时间（NULL = 当前有效）|

### 两种 Snapshot 策略
```sql
-- timestamp 策略：比较 updated_at 字段
strategy='timestamp', updated_at='updated_at'

-- check 策略：比较指定列的值
strategy='check', check_cols=['status', 'email']
-- 或监控所有列
strategy='check', check_cols='all'
```

### 运行
```bash
dbt snapshot                      # 运行所有 snapshots
dbt snapshot --select customers_snapshot
```

---
## 6. 练习题

### Q1 [高频] ref() 函数和直接写表名有什么区别？

<details><summary>参考答案</summary>

- **依赖管理**：`ref()` 自动建立 DAG，保证执行顺序；写死表名则无依赖关系，可能乱序执行
- **环境隔离**：`ref()` 根据 `target.schema` 自动解析到正确环境（dev schema vs prod schema）；写死表名无法隔离
- **跨数据库**：写死名称无法在不同数据仓库（Snowflake/BigQuery）间移植
- **文档和血缘**：`ref()` 被 dbt 解析，自动生成数据血缘图
</details>

---

### Q2 [高频] Incremental 模型 merge 策略如何工作？何时会有问题？

<details><summary>参考答案</summary>

merge 策略执行 `MERGE INTO target USING new_data ON unique_key`：
- 匹配到 unique_key → UPDATE
- 未匹配 → INSERT

**可能的问题**：
1. **不支持 MERGE 的数仓**（老版 Redshift）：改用 `delete+insert`
2. **unique_key 重复**：源数据有重复 key 会导致 merge 行为不确定
3. **删除数据**：merge 不处理源数据中消失的记录（需要额外处理硬删除）
4. **大表 merge 慢**：merge 需要全表扫描匹配，可配合分区减少扫描范围
</details>

---

### Q3 dbt Snapshot 如何实现 SCD Type 2？它存储什么？

<details><summary>参考答案</summary>

dbt Snapshot 定期对比源表与上次快照的差异：
1. 若 `unique_key` 对应行发生变化（timestamp 变了，或 check_cols 值变了）
2. 将旧行的 `dbt_valid_to` 设为当前时间（标记失效）
3. 插入新行，`dbt_valid_from = 当前时间`，`dbt_valid_to = NULL`（当前有效）

查询某个历史时间点的状态：
```sql
SELECT * FROM customers_snapshot
WHERE dbt_valid_from <= '2024-01-15'
  AND (dbt_valid_to > '2024-01-15' OR dbt_valid_to IS NULL)
```
</details>

---

### Q4 如何在 dbt 中处理数据质量问题？有哪些测试工具？

<details><summary>参考答案</summary>

**原生测试（schema.yml 配置）**：
- `unique`, `not_null`, `accepted_values`, `relationships` 四种

**自定义 Generic Test**：在 `tests/generic/` 写 SQL 模板

**Singular Test**：在 `tests/` 写 SQL，返回行数 = 0 则通过

**第三方包**：
- `dbt-expectations`：提供 Great Expectations 风格的测试（行数范围、均值检查等）
- `dbt-utils`：提供 `expression_is_true`、`at_least_one` 等测试

**Source freshness**：`dbt source freshness` 检查数据源是否按时更新
</details>

---

### Q5 [高频] dbt 的 materialization 有哪几种？各自适用场景？

<details><summary>参考答案</summary>

| 类型 | 行为 | 适用场景 |
|------|------|----------|
| `view` | 创建视图，不存储数据 | 轻量转换、常变逻辑 |
| `table` | 每次全量重建表 | 中等数据量、需要索引/集群键 |
| `incremental` | 只处理新增/变更数据 | 大表、高频更新 |
| `ephemeral` | CTE，不物化到数据库 | 中间计算步骤，不需要单独查询 |
| `snapshot` | SCD Type 2 历史记录 | 需要追踪维度变化历史 |

原则：从 `view` 开始，性能不够再改 `table`，数据量大再改 `incremental`。
</details>